In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import requests
import json
import os

data_sources = {
    "sciencekeywords": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/sciencekeywords/?format=json&page_num=1&page_size=2000",
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/sciencekeywords/?format=json&page_num=2&page_size=2000"
        ]
    },
    "platforms": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/platforms/?format=json&page_num=1&page_size=2000",
        ]
    },
    "instruments": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/instruments/?format=json&page_num=1&page_size=2000",
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/instruments/?format=json&page_num=2&page_size=2000",
            ]
    },
    "providers": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/providers/?format=json&page_num=1&page_size=2000",
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/providers/?format=json&page_num=2&page_size=2000"
        ]
    },
    "projects": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/projects/?format=json&page_num=1&page_size=2000"
        ]
    },
    "locations": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/locations/?format=json&page_num=1&page_size=2000"
        ]
    },
    "horizontal_resolution_range": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/horizontalresolutionrange/?format=json&page_num=1&page_size=2000"
        ]
    },
    "vertical_resolution_range": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/verticalresolutionrange/?format=json&page_num=1&page_size=2000"
        ]
    },
    "temporal_resolution_range": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/temporalresolutionrange/?format=json&page_num=1&page_size=2000"
        ]
    },
    "ru_content_type": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/rucontenttype/?format=json&page_num=1&page_size=2000"
        ],
    },
    "data_format": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/DataFormat/?format=json&page_num=1&page_size=2000"
        ],
    },
    "chrono_units": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/chronounits/?format=json&page_num=1&page_size=2000"
        ]
    },
    "mime_type": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/MimeType/?format=json&page_num=1&page_size=2000"
        ]
    },
    "measurement_name": {
        "urls": [
            "https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/MeasurementName/?format=json&page_num=1&page_size=2000"
        ]
    },
}

def get_json(url):
    
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to fetch data from {url} with status code {response.status_code}")
    

def get_pairs(urls):
    pairs = {}
    for url in urls:
        data = get_json(url)
        for item in data.get('concepts', []):
            short_form = item.get('prefLabel', '')
            full_forms = [i.get("text") for i in item.get("definitions", []) if i.get("text")]
            if short_form and full_forms:
                pairs[short_form] = full_forms
    return pairs

In [3]:
data_df = pd.DataFrame(columns=['query', 'context', 'data_source'])
for data_source, data_info in data_sources.items():
    pairs = get_pairs(data_sources[data_source]["urls"])
    pairs = {k: v[0] for k, v in pairs.items() if len(v) > 0}
    df = pd.DataFrame(pairs.items(), columns=['query', 'context'])
    df['data_source'] = data_source

    data_df = pd.concat([data_df, df], ignore_index=True)


data_df.head()
    


,query,context,data_source
0,SEA LEVEL ANOMALY,A sea level anomaly reveals the regional exten...,sciencekeywords
1,SEA SURFACE HEIGHT ANOMALY (SSHA),Difference of sea surface height and mean sea ...,sciencekeywords
2,SEA SURFACE SLOPE,"The slope of the ocean water surface, resultan...",sciencekeywords
3,SEA SURFACE HEIGHT,"The height of the ocean surface above a datum,...",sciencekeywords
4,MEAN SEA SURFACE,The mean sea surface is the displacement of th...,sciencekeywords


In [4]:
data_df.shape

(1751, 3)

# another source of pairs: https://drive.google.com/drive/u/0/folders/1hoaigGg_Bl6kyTe-2VLTEUE3bCtqIyrv

In [5]:
# looping through the folders
pairs = {}
data_df_2 = pd.DataFrame(columns=['query', 'context', 'data_source'])
folder_path = "/rhome/sawale/indus_traning/sentense_transformers/data/Final PIM Vocabulary/"

for p in os.listdir(folder_path):
    # check if the file is json
    if p.endswith('.json'):
        file_path = os.path.join(folder_path, p)
        with open(file_path, 'r') as f:
            data = json.load(f)
        terms = data.get('Terms', [])

        for term in terms:
            short_form = term.get('Term', '')
            full_form = term.get('Definition', '')
            if short_form and full_form:
                pairs[short_form] = full_form
    data_source = p.split("/")[-1].split(".")[0].lower().replace(" ", "_")
    df = pd.DataFrame(pairs.items(), columns=['query', 'context'])
    df['data_source'] = "pim_" + data_source
    data_df_2 = pd.concat([data_df_2, df], ignore_index=True)

data_df_2.shape
        
data_df = pd.concat([data_df, data_df_2], ignore_index=True)

data_df.shape

(75688, 3)

In [6]:
data_df["data_source"].value_counts()

data_source
pim_ceos_instruments                              7107
pim_spase_instruments                             6784
pim_lsda_missions                                 4380
pim_oscar_missions                                4380
pim_gcmd_platforms                                4380
pim_astronomy_and_astrophysics_flight_missions    3647
pim_pds_data_dictionary_instruments               3620
pim_pds_data_dictionary_platforms                 3620
pim_lsda_platforms                                3620
pim_pds_data_dictionary_missions                  3620
pim_oscar_instruments                             3620
pim_planetary_missions_beyond_earth_orbit         3620
pim_beyond_earth_missions                         3522
providers                                         3423
pim_spase_observatories                           3401
sciencekeywords                                   2602
projects                                          1768
instruments                                       175

In [7]:
data_df.head()

,query,context,data_source
0,SEA LEVEL ANOMALY,A sea level anomaly reveals the regional exten...,sciencekeywords
1,SEA SURFACE HEIGHT ANOMALY (SSHA),Difference of sea surface height and mean sea ...,sciencekeywords
2,SEA SURFACE SLOPE,"The slope of the ocean water surface, resultan...",sciencekeywords
3,SEA SURFACE HEIGHT,"The height of the ocean surface above a datum,...",sciencekeywords
4,MEAN SEA SURFACE,The mean sea surface is the displacement of th...,sciencekeywords


In [8]:
data_df.shape, data_df["context"].nunique(), data_df["query"].nunique()

((75688, 3), 16715, 17727)

In [10]:
data_df["context"].nunique()

# drop duplicates based on context
data_df = data_df.drop_duplicates(subset=['context'])

data_df.shape

(16716, 3)

# convert this pairs as jsonl file

In [4]:
def create_training_files(df: pd.DataFrame, output_dir: str = '_test_output'):
    """
    Generates corpus, queries, and qrels files from a DataFrame.

    The function creates files in the specified output directory:
    - corpus.jsonl: Contains unique contexts with assigned IDs.
    - queries.jsonl: Contains unique queries with assigned IDs.
    - qrels/ (folder): Contains multiple .tsv files, one for each data_source,
      mapping query IDs to relevant corpus IDs.

    Args:
        df (pd.DataFrame): A DataFrame with 'query', 'context', and 'data_source' columns.
        output_dir (str): The directory to save the generated files.
    """
    # --- 1. Input Validation ---
    if not all(col in df.columns for col in ['query', 'context', 'data_source']):
        raise ValueError("Input DataFrame must contain 'query', 'context', and 'data_source' columns.")

    if df.isnull().values.any():
        print("Warning: DataFrame contains NaN values. Rows with NaN will be dropped.")
        df.dropna(subset=['query', 'context', 'data_source'], inplace=True)

    # --- 2. Create Output Directory ---
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # --- 3. Process Corpus ---
    # Get unique contexts and assign a document ID (doc_id) to each.
    unique_contexts = df['context'].unique()
    context_to_id = {context: f"doc_{i}" for i, context in enumerate(unique_contexts)}
    
    corpus_filepath = os.path.join(output_dir, 'corpus.jsonl')
    with open(corpus_filepath, 'w', encoding='utf-8') as f_corpus:
        for context, doc_id in context_to_id.items():
            record = {
                "_id": doc_id,
                "text": context,
                "title": ""  # Title is often empty if not available
            }
            f_corpus.write(json.dumps(record) + '\n')
    print(f"Corpus file created at: {corpus_filepath}")

    # --- 4. Process Queries ---
    # Get unique queries and assign a query ID (qid) to each.
    unique_queries = df['query'].unique()
    query_to_id = {query: f"q_{i}" for i, query in enumerate(unique_queries)}

    queries_filepath = os.path.join(output_dir, 'queries.jsonl')
    with open(queries_filepath, 'w', encoding='utf-8') as f_queries:
        for query, qid in query_to_id.items():
            record = {
                "_id": qid,
                "text": query
            }
            f_queries.write(json.dumps(record) + '\n')
    print(f"Queries file created at: {queries_filepath}")

    # --- 5. Create Qrels (Query-Relevance pairs) based on data_source ---
    # Create a directory for the qrels files
    qrels_dir = os.path.join(output_dir, 'qrels')
    if not os.path.exists(qrels_dir):
        os.makedirs(qrels_dir)
        print(f"Created directory: {qrels_dir}")

    # Group the dataframe by the data_source and create a file for each
    for source, group_df in df.groupby('data_source'):
        qrels_filepath = os.path.join(qrels_dir, f"{source}.tsv")
        with open(qrels_filepath, 'w', encoding='utf-8') as f_qrels:
            # Write the header for the TSV file
            f_qrels.write("query-id\tcorpus-id\tscore\n")
            
            # Iterate through the rows of the group to create the mappings
            for _, row in group_df.iterrows():
                query = row['query']
                context = row['context']
                
                # Get the corresponding IDs
                qid = query_to_id[query]
                doc_id = context_to_id[context]
                
                # Write the mapping to the file. Score is 1 for positive pairs.
                f_qrels.write(f"{qid}\t{doc_id}\t1\n")
        print(f"Qrels file for source '{source}' created at: {qrels_filepath}")
        
    print("\nProcessing complete.")



In [5]:
create_training_files(data_df, output_dir='/rhome/sawale/indus_traning/sentense_transformers/data/short_full_form_pairs_v1')

Created directory: /rhome/sawale/indus_traning/sentense_transformers/data/short_full_form_pairs_v1
Corpus file created at: /rhome/sawale/indus_traning/sentense_transformers/data/short_full_form_pairs_v1/corpus.jsonl
Queries file created at: /rhome/sawale/indus_traning/sentense_transformers/data/short_full_form_pairs_v1/queries.jsonl
Created directory: /rhome/sawale/indus_traning/sentense_transformers/data/short_full_form_pairs_v1/qrels
Qrels file for source 'chrono_units' created at: /rhome/sawale/indus_traning/sentense_transformers/data/short_full_form_pairs_v1/qrels/chrono_units.tsv
Qrels file for source 'data_format' created at: /rhome/sawale/indus_traning/sentense_transformers/data/short_full_form_pairs_v1/qrels/data_format.tsv
Qrels file for source 'instruments' created at: /rhome/sawale/indus_traning/sentense_transformers/data/short_full_form_pairs_v1/qrels/instruments.tsv
Qrels file for source 'locations' created at: /rhome/sawale/indus_traning/sentense_transformers/data/short_f

# test the jsonl files

In [1]:

from datasets import load_dataset

path = "/rhome/sawale/indus_traning/sentense_transformers/data/short_full_form_pairs"
# Load the dataset to verify
corpus = load_dataset(
    path,
    data_files="corpus.jsonl",
    split="train",
)
queries = load_dataset(
    path,
    data_files="queries.jsonl",
    split="train",
)

relevant_docs_data = load_dataset(path,
                                  data_files="qrels/instruments.tsv")


In [3]:
corpus, queries, relevant_docs_data

(Dataset({
     features: ['_id', 'text', 'title'],
     num_rows: 16713
 }),
 Dataset({
     features: ['_id', 'text'],
     num_rows: 16210
 }),
 DatasetDict({
     train: Dataset({
         features: ['query-id', 'corpus-id', 'score'],
         num_rows: 1733
     })
 }))

In [6]:
data_files = [
            "qrels/chrono_units.tsv",
            "qrels/data_format.tsv",
            "qrels/instruments.tsv",
            "qrels/locations.tsv",
            "qrels/measurement_name.tsv",
            "qrels/mime_type.tsv",
            "qrels/platforms.tsv",
            "qrels/projects.tsv",
            "qrels/providers.tsv",
            "qrels/ru_content_type.tsv",
            "qrels/sciencekeywords.tsv",
            "qrels/temporal_resolution_range.tsv",
            "qrels/pim_astronomy_and_astrophysics_flight_missions.tsv",
            "qrels/pim_beyond_earth_missions.tsv",
            "qrels/pim_ceos_instruments.tsv",
            "qrels/pim_ceos_missions.tsv",
            "qrels/pim_gcmd_instruments.tsv",
            "qrels/pim_gcmd_platforms.tsv",
            "qrels/pim_high_energy_astrophysics_missions.tsv",
            "qrels/pim_mast_missions.tsv",
            "qrels/pim_nasa_heliophysics_sun-planet_missions.tsv",
            "qrels/pim_pds_mission_archive_page.tsv",
            "qrels/pim_planetary_missions_beyond_earth_orbit.tsv",
            "qrels/pim_spase_instruments.tsv",
            "qrels/pim_spase_observatories.tsv"
        ]


dist = {}
for i in data_files:
    relevant_docs_data = load_dataset(path, data_files=i, split="train")
    dist[i] = len(relevant_docs_data)

dist

{'qrels/chrono_units.tsv': 158,
 'qrels/data_format.tsv': 137,
 'qrels/instruments.tsv': 1733,
 'qrels/locations.tsv': 20,
 'qrels/measurement_name.tsv': 35,
 'qrels/mime_type.tsv': 32,
 'qrels/platforms.tsv': 1100,
 'qrels/projects.tsv': 1747,
 'qrels/providers.tsv': 3386,
 'qrels/ru_content_type.tsv': 85,
 'qrels/sciencekeywords.tsv': 2558,
 'qrels/temporal_resolution_range.tsv': 2,
 'qrels/pim_astronomy_and_astrophysics_flight_missions.tsv': 26,
 'qrels/pim_beyond_earth_missions.tsv': 133,
 'qrels/pim_ceos_instruments.tsv': 301,
 'qrels/pim_ceos_missions.tsv': 190,
 'qrels/pim_gcmd_instruments.tsv': 450,
 'qrels/pim_gcmd_platforms.tsv': 369,
 'qrels/pim_high_energy_astrophysics_missions.tsv': 32,
 'qrels/pim_mast_missions.tsv': 18,
 'qrels/pim_nasa_heliophysics_sun-planet_missions.tsv': 23,
 'qrels/pim_pds_mission_archive_page.tsv': 28,
 'qrels/pim_planetary_missions_beyond_earth_orbit.tsv': 141,
 'qrels/pim_spase_instruments.tsv': 2305,
 'qrels/pim_spase_observatories.tsv': 1704}

In [15]:
relevant_docs_data

DatasetDict({
    train: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 1733
    })
})

In [9]:
# relevant_docs_data["train"].to_pandas().head()


relevant_docs_data = load_dataset(
    path,
    split="train",
    data_files="qrels/instruments.tsv",
)

In [10]:
relevant_docs_data

Dataset({
    features: ['query-id', 'corpus-id', 'score'],
    num_rows: 1751
})